# Introduction to catsim

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/01_introduction_to_catsim.ipynb)

`catsim` is a toolkit for computerized adaptive testing (CAT). The modern
package architecture has four main pieces:

- `ItemBank`: calibrated item parameters and item-level utilities
- `CatEngine`: stepwise CAT execution for manual or third-party use
- `SimulationRunner`: repeated CAT sessions for research studies
- `SimulationResult`: aggregate outputs from a simulation run

This notebook is the shortest path through those ideas. It shows one small
simulation, one inspected session, and one compact manual CAT loop.

## Installation

Install the package with:

```bash
pip install -U catsim
```

If you are using Colab, restart the runtime after installation if imports
fail in the first execution.

## Setup

We will keep the examples small, deterministic, and fast enough to run in a
notebook session.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from catsim import plot
from catsim.engine import CatEngine, RunContext
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import FixedPointInitializer, UniformInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## Generate an item bank

A CAT needs calibrated items. In `catsim`, an item bank stores the item
parameters and a few precomputed values that only depend on the items
themselves.

In [ ]:
item_bank = ItemBank.generate(120, model="4PL", rng=7)
print(f"Bank size: {item_bank.n_items}")
print("First three items [a, b, c, d]:")
print(np.round(item_bank.items[:3], 3))

In [ ]:
item = item_bank.items[0]
plot.item_curve(item[0], item[1], item[2], item[3], title="One Item")
plt.show()
plt.close("all")

## Run a small simulation

A `CatEngine` combines the item bank with an initializer, a selector, an
estimator and a stopper. A `SimulationRunner` runs one simulated CAT session on
that engine for each true ability it is given, and returns a `SimulationResult`.
Here the 30 examinees' true abilities are drawn from a standard normal
distribution, and `rng=19` seeds the run's random draws, so the run gives the
same result every time.

The stopper here is a `StopRule` that combines a precision rule with a length
cap: each session ends once the standard error of the ability estimate is at
most 0.4, or at 40 items, and never before 6 items, so test length varies from
examinee to examinee.

In [ ]:
max_items = 40
stopper = StopRule(StandardErrorStopper(0.4), MaxLengthStopper(max_items), min_items=6)
engine = CatEngine(
  item_bank=item_bank,
  initializer=UniformInitializer(),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=stopper,
)
true_thetas = np.random.default_rng(5).normal(0, 1, 30)

result = SimulationRunner(engine).run(true_thetas, rng=19)
lengths = result.test_lengths

print(f"Sessions: {len(result.sessions)}")
print(f"Bias: {result.bias:.3f}")
print(f"RMSE: {result.rmse:.3f}")
print(f"Test length: min {min(lengths)}, mean {np.mean(lengths):.2f}, max {max(lengths)}")
print(f"Sessions that ran to the {max_items}-item cap: {np.count_nonzero(lengths == max_items)}")

Most sessions stop as soon as the standard error falls below 0.4. The six that
run to the cap all belong to examinees about one unit or more from the middle of
the ability scale, where the bank has fewer informative items.

## Inspect one simulated session

`SimulationResult.sessions` gives access to the individual CAT traces. Each
session is a `CatSessionState`.

In [ ]:
session = result.sessions[0]
print("Session id:", session.session_id)
print("True theta:", round(float(session.true_theta), 3))
print("Final theta:", round(session.current_theta, 3))
print("Items administered:", [int(item) for item in session.administered_item_ids[:8]], "...")
print("Responses:", session.responses[:8], "...")
print("Theta history:", [round(float(theta), 3) for theta in session.theta_history[:6]], "...")

## Plot one session and the item exposure profile

The upper panel follows the first session: the estimate $\hat{\theta}$ moves
toward the true $\theta$ (dashed line), the difficulty of each new item follows
the estimate, and the test information $I(\theta)$ grows while the standard
error $SEE$ shrinks, until $SEE$ falls below 0.4 and the session stops. The
lower panel shows each item's exposure rate (the share of sessions that
administered it), with items ordered by difficulty: maximum-information
selection gives some items to most examinees and never uses others.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 8))
plot.test_progress(simulation=result, index=0, info=True, see=True, ax=axes[0])
plot.item_exposure(simulation=result, par="b", ax=axes[1])
plt.tight_layout()
plt.show()
plt.close("all")

## Run one CAT session manually

`CatEngine` is the right entry point when another application controls the
session and provides real responses. `select_next` returns the next item, or
`None` once the session has ended, so the loop needs no other stop check.

In [ ]:
engine = CatEngine(
  item_bank=item_bank,
  initializer=FixedPointInitializer(0.0),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=MaxLengthStopper(4),
)

context = RunContext(rng=np.random.default_rng(123))
manual_session = engine.start_session(session_id=1001, context=context)

mocked_responses = [True, False, True, True]
while (next_item := engine.select_next(manual_session, context)) is not None:
  response = mocked_responses[manual_session.administered_count]
  step = engine.apply_response(manual_session, next_item, response, context)
  print(
    f"item={step.selected_item_id:>3} response={step.response} theta={step.updated_theta: .3f} stopped={step.stopped}"
  )

print("Final manual theta:", round(manual_session.current_theta, 3))
print("Stop reason:", manual_session.stop_reason)

## Where to go next

- `02_manual_cat_sessions.ipynb` focuses on external-response workflows.
- `03_simulation_studies.ipynb` shows research-oriented experiments.
- `04_item_selection_and_stopping.ipynb` compares CAT strategies.
- `05_visualization_and_analysis.ipynb` focuses on figures and interpretation.